In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import numpy as np
import matplotlib.pyplot as plt

import networkx as nx
from networkx.algorithms import smallworld
import random
from collections import Counter
from scipy.spatial import cKDTree

from tqdm import tqdm

import pandas as pd
import seaborn as sns

In [9]:
from src.neuron_population import NeuronPopulation
from src.connectome import Connectome
from src.overhead import Simulation
from src.neuron_templates import neuron_type_IZ
from src.network_grower import *
from src.network_generators import *
from src.neuron_type_distributor import *
from src.network_weight_distributor import *
from src.external_inputs import *

## DF

In [12]:
# df = pd.read_csv("higher_sweep.csv")
# df = pd.read_csv("week12_normalize_target_sweep.csv")
# df = pd.read_csv("week12_normalize_target_bfactor_sweep.csv")
df_orig = pd.read_csv("results/threshold_heterogeneity_bifurcation_sweep.csv")
# df = pd.read_csv("week12_normalize_target_random_topology_sweep.csv")

In [17]:
df = df_orig.drop(columns=["job_index", "num_jobs", "generator_name"])
df

,topology,topology_seed,variance_ss4_vt,variance_b_vt,truncate_std,heterogeneity_param_index,heterogeneity_param_name,n_neurons,fixed_indegree,dt_ms,...,participation_frac_mean_300ms,participation_frac_median_300ms,participation_frac_p95_300ms,participation_frac_total,participation_frac_total_E,participation_frac_total_I,psd_peak_freq_hz,psd_peak_amplitude,psd_peak_ratio,pop_spec_entropy
0,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.002333,0.002,0.0029,0.007,0.00000,0.035,83.991601,3.187517e+01,7.029057,8.131760
1,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.001667,0.002,0.0020,0.006,0.00000,0.030,62.993701,2.507280e+01,6.401692,8.137224
2,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.677333,0.691,0.7009,0.931,0.91500,0.995,10.000000,1.955452e+06,285.456411,7.650227
3,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.700667,0.702,0.7344,0.941,0.92750,0.995,13.000000,3.515773e+06,407.421205,7.353548
4,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.700667,0.724,0.7285,0.968,0.96000,1.000,8.000000,1.919869e+06,210.825284,7.599680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.576667,0.572,0.5990,0.863,0.82875,1.000,16.998300,3.569967e+05,56.710037,7.918951
2696,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.559333,0.569,0.5744,0.845,0.80625,1.000,5.999400,5.012505e+05,75.749253,7.945796
2697,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.570000,0.566,0.5768,0.834,0.79250,1.000,5.999400,6.907224e+05,95.103630,7.909024
2698,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.563000,0.570,0.5862,0.849,0.81125,1.000,6.999300,4.800957e+05,70.333539,7.929911


In [18]:
df_down = df[df["sweep_direction"] == "down"]
df_down

,topology,topology_seed,variance_ss4_vt,variance_b_vt,truncate_std,heterogeneity_param_index,heterogeneity_param_name,n_neurons,fixed_indegree,dt_ms,...,participation_frac_mean_300ms,participation_frac_median_300ms,participation_frac_p95_300ms,participation_frac_total,participation_frac_total_E,participation_frac_total_I,psd_peak_freq_hz,psd_peak_amplitude,psd_peak_ratio,pop_spec_entropy
9,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.588333,0.588,0.5997,0.782,0.75625,0.885,25.0000,2.184059e+06,144.603081,7.908782
10,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.638333,0.641,0.6473,0.848,0.81875,0.965,21.0000,5.797364e+05,59.804212,7.967406
11,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.699333,0.697,0.7132,0.952,0.94250,0.990,10.9989,2.373758e+06,272.482920,7.555323
12,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.645000,0.638,0.6884,0.929,0.91500,0.985,2.9997,1.449373e+06,172.077355,7.190897
13,spatial,1234,0.0,0.0,2.0,6,Vt,1000,100,0.1,...,0.681667,0.694,0.7147,0.958,0.94875,0.995,7.9992,1.790782e+06,214.762660,7.705367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.576667,0.572,0.5990,0.863,0.82875,1.000,16.9983,3.569967e+05,56.710037,7.918951
2696,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.559333,0.569,0.5744,0.845,0.80625,1.000,5.9994,5.012505e+05,75.749253,7.945796
2697,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.570000,0.566,0.5768,0.834,0.79250,1.000,5.9994,6.907224e+05,95.103630,7.909024
2698,fixed,1236,7.0,7.0,2.0,6,Vt,1000,100,0.1,...,0.563000,0.570,0.5862,0.849,0.81125,1.000,6.9993,4.800957e+05,70.333539,7.929911
